In [ ]:
import os, re, json, itertools, warnings
from collections import Counter
from datetime import datetime
import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
import networkx as nx
from tqdm.auto import tqdm
from sklearn.feature_extraction.text import TfidfVectorizer, CountVectorizer
from wordcloud import WordCloud
import nltk

warnings.filterwarnings('ignore')
try:
    from nltk.corpus import stopwords as set_stopwords
    PT_NLTK = set(set_stopwords.words('portuguese'))
except LookupError:
    nltk.download('stopwords', quiet=True)
    from nltk.corpus import stopwords as set_stopwords
    PT_NLTK = set(set_stopwords.words('portuguese'))

BASE_DIR = r''
CLEAN_PATH = os.path.join(BASE_DIR, 'Data', 'dataset_clean_cnj.json')
RESULTS_JSON = os.path.join(BASE_DIR, 'Results', 'eda_content_results.json')
FIG_DIR = os.path.join(BASE_DIR, 'Results', 'Figures', 'Content_Individual')
os.makedirs(FIG_DIR, exist_ok=True)

matplotlib.rcParams.update({'figure.dpi': 300, 'savefig.dpi': 300, 'axes.spines.top': False, 'axes.spines.right': False})

FIELDS = ['inteiro_teor', 'fato', 'direito', 'pedido']
FIELDS_LABEL = ['Full Text', 'Facts', 'Legal Basis', 'Legal Claim']
FIELD_COLORS = ['#1B4F72', '#C0392B', '#117A65', '#784212']
PALETTE = ['#2C6E49', '#C77DFF']

PII_TOKENS = {'[cpf]', '[cnpj]', '[processo_cnj]', '[oab]', '[cep]', '[telefone]', '[processo]', '[valor_monetario]', '[data]', 
              '[pessoa_física]', '[instituição]', '[município/estado]', '[ator_juridico]', '[papel_processual]'}
LEGAL_STOPWORDS = {'de', 'da', 'do', 'nos', 'nas', 'ao', 'aos', 'um', 'que', 'com', 'por', 'para', 'pela', 'pelo', 'sobre', 
                   'como', 'nao', 'mas', 'ser', 'ter', 'foi', 'tem', 'deve', 'processo', 'presente', 'parte', 'partes', 'fls'}
STOPWORDS = PT_NLTK | LEGAL_STOPWORDS | PII_TOKENS

RESULTS = {'metadata': {'stage': 2, 'started_at': datetime.now().isoformat()}}

with open(CLEAN_PATH, 'r', encoding='utf-8') as f:
    df = pd.DataFrame(json.load(f))
print(f'Loaded {len(df)} records from clean dataset.')

## Section 1: Tokenization and Vocabulary Audit


In [ ]:
def tokenize(text):
    text = str(text).lower()
    text = re.sub(r'\[.*?\]', ' ', text) 
    text = re.sub(r'\b\d+\b', ' ', text) 
    text = re.sub(r'[^\w\s]', ' ', text)
    return [t for t in text.split() if len(t) >= 4 and t not in STOPWORDS]

vocab_stats = {}
for fld, lbl in zip(FIELDS, FIELDS_LABEL):
    df[f'{fld}_tokens'] = df[fld].apply(tokenize)
    all_tokens = list(itertools.chain.from_iterable(df[f'{fld}_tokens']))
    total = len(all_tokens)
    unique = len(set(all_tokens))
    vocab_stats[fld] = {'total': total, 'unique': unique, 'ttr': unique/total if total else 0}
    print(f'{lbl}: Total={total:,} | Unique={unique:,} | Lexical Richness={unique/total:.4f}')
RESULTS['vocabulary'] = vocab_stats

## Section 2: Word Frequency & TF-IDF (Individual Figures)


In [ ]:
freq_results = {}
tfidf_results = {}

for fld, lbl, color in zip(FIELDS, FIELDS_LABEL, FIELD_COLORS):
    tokens = list(itertools.chain.from_iterable(df[f'{fld}_tokens']))
    top_freq = Counter(tokens).most_common(20)
    freq_results[fld] = dict(top_freq)
    
    fig, ax = plt.subplots(figsize=(6, 5))
    words, counts = zip(*top_freq)
    ax.barh(words[::-1], counts[::-1], color=color, alpha=0.8)
    ax.set_title(f'Raw Frequency: {lbl}', weight='bold')
    ax.set_xlabel('Occurrences')
    p = os.path.join(FIG_DIR, f'fig1_freq_{fld}.png')
    fig.savefig(p, bbox_inches='tight')
    plt.close()

    corpus = df[fld].astype(str).tolist()
    vec = TfidfVectorizer(max_features=2000, stop_words=list(STOPWORDS), token_pattern=r'(?u)\b[a-zA-Z\u00C0-\u024F]{4,}\b', sublinear_tf=True)
    matrix = vec.fit_transform(tqdm(corpus, desc=f'TF-IDF {lbl}', leave=False))
    mean_scores = np.asarray(matrix.mean(axis=0)).flatten()
    top_idx = mean_scores.argsort()[::-1][:20]
    top_tfidf_words = vec.get_feature_names_out()[top_idx]
    
    tfidf_results[fld] = dict(zip(top_tfidf_words, mean_scores[top_idx]))
    fig, ax = plt.subplots(figsize=(6, 5))
    ax.barh(top_tfidf_words[::-1], mean_scores[top_idx][::-1], color=color, alpha=0.85)
    ax.set_title(f'TF-IDF Importance: {lbl}', weight='bold')
    p = os.path.join(FIG_DIR, f'fig2_tfidf_{fld}.png')
    fig.savefig(p, bbox_inches='tight')
    plt.close()

RESULTS['word_frequency'] = freq_results
RESULTS['tfidf_top_terms'] = tfidf_results

## Section 3: Word Clouds (Individual Exports)


In [ ]:
for i, (fld, lbl) in enumerate(zip(FIELDS, FIELDS_LABEL)):
    freq = Counter(list(itertools.chain.from_iterable(df[f'{fld}_tokens'])))
    wc = WordCloud(colormap='viridis' if i%2==0 else 'plasma', width=800, height=400, background_color='white', max_words=100)
    wc.generate_from_frequencies(freq)
    
    fig, ax = plt.subplots(figsize=(8, 4))
    ax.imshow(wc, interpolation='bilinear')
    ax.axis('off')
    ax.set_title(f'Word Cloud: {lbl}', weight='bold', pad=10)
    fig.savefig(os.path.join(FIG_DIR, f'fig3_wc_{fld}.png'), bbox_inches='tight')
    plt.close()
    
    if fld in ['direito', 'pedido']:
        for is_rec, dtype in [(False, 'First Instance'), (True, 'Appeal')]:
            subset = df[df.is_recurso == is_rec]
            f = Counter(list(itertools.chain.from_iterable(subset[f'{fld}_tokens'])))
            wc = WordCloud(colormap='Blues_r' if not is_rec else 'Oranges_r', width=800, height=400, background_color='white', max_words=80)
            wc.generate_from_frequencies(f)
            fig, ax = plt.subplots(figsize=(8, 4))
            ax.imshow(wc, interpolation='bilinear')
            ax.axis('off')
            ax.set_title(f'{lbl} - {dtype}', weight='bold', pad=10)
            fig.savefig(os.path.join(FIG_DIR, f'fig4_wc_{fld}_{dtype.replace(" ", "")}.png'), bbox_inches='tight')
            plt.close()

## Section 4: Lexical Richness (Type-Token Ratio Violin Plots by Field)


In [ ]:
ttr_stats = {}
for fld, lbl in zip(FIELDS, FIELDS_LABEL):
    df[f'{fld}_ttr'] = df[f'{fld}_tokens'].apply(lambda x: len(set(x))/len(x) if x else 0)
    
    d0 = df[df.is_recurso == False][f'{fld}_ttr']
    d1 = df[df.is_recurso == True][f'{fld}_ttr']
    ttr_stats[fld] = {'FirstInstance_mean': float(d0.mean()), 'Appeal_mean': float(d1.mean())}
    
    fig, ax = plt.subplots(figsize=(5, 5))
    parts = ax.violinplot([d0, d1], showmedians=True)
    parts['bodies'][0].set_facecolor(PALETTE[0])
    parts['bodies'][1].set_facecolor(PALETTE[1])
    for b in parts['bodies']: b.set_alpha(0.75)
    parts['cmedians'].set_color('black')
    
    ax.set_xticks([1, 2]); ax.set_xticklabels(['First Instance', 'Appeal'])
    ax.set_ylabel('Type-Token Ratio')
    ax.set_title(f'Lexical Diversity: {lbl}', weight='bold')
    fig.savefig(os.path.join(FIG_DIR, f'fig5_ttr_{fld}.png'), bbox_inches='tight')
    plt.close()
RESULTS['lexical_richness'] = ttr_stats

## Section 5: Sentence Dynamics (Density Histograms)


In [ ]:
SENT_SPLIT = re.compile(r'(?<=[.!?;])\s+')
for fld, lbl, color in zip(FIELDS, FIELDS_LABEL, FIELD_COLORS):
    df[f'{fld}_sents'] = df[fld].astype(str).str.split(SENT_SPLIT).str.len()
    p99 = df[f'{fld}_sents'].quantile(0.99)
    
    fig, ax = plt.subplots(figsize=(6, 4))
    d0 = df[df.is_recurso == False][f'{fld}_sents'].clip(upper=p99)
    d1 = df[df.is_recurso == True][f'{fld}_sents'].clip(upper=p99)
    ax.hist(d0, bins=35, alpha=0.6, density=True, label='First Instance', color=PALETTE[0])
    ax.hist(d1, bins=35, alpha=0.6, density=True, label='Appeal', color=PALETTE[1])
    ax.set_xlabel('Number of Sentences')
    ax.set_title(f'Sentence Count: {lbl}', weight='bold')
    ax.legend()
    fig.savefig(os.path.join(FIG_DIR, f'fig6_sent_len_{fld}.png'), bbox_inches='tight')
    plt.close()

## Section 6: Legal Article Network Graph (The Improvement)


In [ ]:
import networkx as nx
ART_RE = re.compile(r'art(?:igo)?[\s\.]*?(\d{1,4})', re.IGNORECASE)
df['direito_arts'] = df['direito'].astype(str).apply(lambda x: list(set(ART_RE.findall(x))))

all_arts = list(itertools.chain.from_iterable(df['direito_arts']))
art_counts = Counter(all_arts)
top_arts = [a for a, c in art_counts.most_common(20)]

G = nx.Graph()
for arts in df['direito_arts']:
    valid = [a for a in arts if a in top_arts]
    for a, b in itertools.combinations(valid, 2):
        if G.has_edge(a, b):
            G[a][b]['weight'] += 1
        else:
            G.add_edge(a, b, weight=1)
            
fig, ax = plt.subplots(figsize=(10, 10))
pos = nx.spring_layout(G, k=0.5, iterations=50)
weights = [G[u][v]['weight'] * 0.1 for u, v in G.edges()]
degrees = [G.degree(n) * 300 for n in G.nodes()]

nx.draw_networkx_nodes(G, pos, ax=ax, node_size=degrees, node_color='#8E44AD', alpha=0.8)
nx.draw_networkx_edges(G, pos, ax=ax, width=weights, edge_color='#BDC3C7', alpha=0.6)
nx.draw_networkx_labels(G, pos, ax=ax, labels={n: f'Art. {n}' for n in G.nodes()}, font_size=10, font_weight='bold')
ax.set_title('Top Legal Articles Co-occurrence Network', weight='bold', size=14)
ax.axis('off')
p = os.path.join(FIG_DIR, 'fig7_article_network_graph.png')
fig.savefig(p, bbox_inches='tight')
plt.close()

RESULTS['article_citations'] = {'top_20': {f'Art. {k}': v for k, v in art_counts.most_common(20)}}

## Section 7: Export Checkpoint


In [ ]:
with open(RESULTS_JSON, 'w', encoding='utf-8') as f:
    json.dump(RESULTS, f, indent=2, ensure_ascii=False)
print('\nContent EDA Complete. All independent PNGs exported.')
print(f'Results Checkpoint saved: {RESULTS_JSON}')

In [ ]:
TABLE_DIR = os.path.join(BASE_DIR, 'Results', 'Tables')
os.makedirs(TABLE_DIR, exist_ok=True)

content_card = {
    'n_total': int(len(df)),
    'n_non_appeal': int((df['is_recurso'] == False).sum()),
    'n_appeal': int((df['is_recurso'] == True).sum()),
    'vocabulary': RESULTS.get('vocabulary', {}),
    'lexical_richness': RESULTS.get('lexical_richness', {}),
    'article_citations': RESULTS.get('article_citations', {})
}

rows = []
for fld in FIELDS:
    rows.append({
        'field': fld,
        'unique_terms': int(RESULTS['vocabulary'][fld]['unique']),
        'total_terms': int(RESULTS['vocabulary'][fld]['total']),
        'type_token_ratio': float(RESULTS['vocabulary'][fld]['ttr']),
        'mean_ttr_first_instance': float(RESULTS['lexical_richness'][fld]['FirstInstance_mean']),
        'mean_ttr_appeal': float(RESULTS['lexical_richness'][fld]['Appeal_mean']),
    })

df_card = pd.DataFrame(rows)
csv_path = os.path.join(TABLE_DIR, 'content_dataset_card.csv')
json_path = os.path.join(BASE_DIR, 'Results', 'content_dataset_card.json')

df_card.to_csv(csv_path, index=False, encoding='utf-8-sig')
with open(json_path, 'w', encoding='utf-8') as f:
    json.dump(content_card, f, indent=2, ensure_ascii=False)

print('\n=== CONTENT DATASET CARD ===')
print(df_card.to_string(index=False))
print(f'\nSaved CSV  -> {csv_path}')
print(f'Saved JSON -> {json_path}')